# Exploratory Data Analysis (EDA) - Football Match Results

This notebook performs comprehensive EDA on the football match results dataset to understand patterns, identify gaps, and prepare for further analysis.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Load the dataset
df = pd.read_csv('results.csv')
print(f"Dataset loaded successfully with {df.shape[0]} rows and {df.shape[1]} columns")

## 1. Initial Data Exploration

In [ ]:
# Display first few rows
print("First 5 rows of the dataset:")
df.head()

In [ ]:
# Dataset info
print("Dataset Information:")
print(df.info())
print("\n" + "="*50 + "\n")
print("Basic Statistics:")
df.describe()

In [ ]:
# Check for missing values
print("Missing Values Count:")
missing_values = df.isnull().sum()
missing_percent = (missing_values / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing_values, 'Percentage': missing_percent})
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# Check for duplicates
print(f"Number of duplicate rows: {df.duplicated().sum()}")

## 2. Data Quality Assessment & Gap Identification

In [ ]:
# Convert date column to datetime
df['date'] = pd.to_datetime(df['date'])

# Check date range
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Total years covered: {(df['date'].max() - df['date'].min()).days / 365.25:.1f} years")

In [ ]:
# Analyze score columns for potential issues
print("Score Analysis:")
print(f"Matches with 0 total goals: {(df['home_score'] + df['away_score'] == 0).sum()}")
print(f"Matches with negative scores: {((df['home_score'] < 0) | (df['away_score'] < 0)).sum()}")
print(f"Maximum goals in a single match: {(df['home_score'] + df['away_score']).max()}")

In [ ]:
# Check neutral venue distribution
print("Neutral Venue Distribution:")
print(df['neutral'].value_counts())
print(f"\nPercentage of neutral matches: {(df['neutral'] == 'TRUE').sum() / len(df) * 100:.2f}%")

## 3. Temporal Analysis

In [ ]:
# Extract time-based features
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day_of_week'] = df['date'].dt.dayofweek

# Matches per year
plt.figure(figsize=(14, 6))
matches_per_year = df['year'].value_counts().sort_index()
plt.plot(matches_per_year.index, matches_per_year.values, linewidth=1)
plt.title('Number of Matches per Year')
plt.xlabel('Year')
plt.ylabel('Number of Matches')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Average goals per year
df['total_goals'] = df['home_score'] + df['away_score']
goals_per_year = df.groupby('year')['total_goals'].mean()

plt.figure(figsize=(14, 6))
plt.plot(goals_per_year.index, goals_per_year.values, linewidth=1, color='red')
plt.title('Average Goals per Match per Year')
plt.xlabel('Year')
plt.ylabel('Average Goals')
plt.grid(True, alpha=0.3)
plt.show()

## 4. Team & Geographic Analysis

In [ ]:
# Top 20 most active teams
plt.figure(figsize=(12, 8))
teams = pd.concat([df['home_team'], df['away_team']])
top_teams = teams.value_counts().head(20)
top_teams.plot(kind='barh')
plt.title('Top 20 Most Active Teams')
plt.xlabel('Number of Matches')
plt.tight_layout()
plt.show()

In [ ]:
# Top countries hosting matches
plt.figure(figsize=(12, 8))
top_countries = df['country'].value_counts().head(15)
top_countries.plot(kind='barh')
plt.title('Top 15 Countries Hosting Matches')
plt.xlabel('Number of Matches')
plt.tight_layout()
plt.show()

## 5. Tournament Analysis

In [ ]:
# Tournament distribution
plt.figure(figsize=(12, 8))
tournament_counts = df['tournament'].value_counts().head(15)
tournament_counts.plot(kind='barh')
plt.title('Top 15 Tournament Types')
plt.xlabel('Number of Matches')
plt.tight_layout()
plt.show()

In [ ]:
# Average goals by tournament type
tournament_goals = df.groupby('tournament')['total_goals'].mean().sort_values(ascending=False).head(15)

plt.figure(figsize=(12, 8))
tournament_goals.plot(kind='barh', color='green')
plt.title('Average Goals per Match by Tournament Type')
plt.xlabel('Average Goals')
plt.tight_layout()
plt.show()

## 6. Performance Analysis

In [ ]:
# Home vs Away performance
home_wins = (df['home_score'] > df['away_score']).sum()
away_wins = (df['away_score'] > df['home_score']).sum()
draws = (df['home_score'] == df['away_score']).sum()

plt.figure(figsize=(8, 6))
labels = ['Home Wins', 'Away Wins', 'Draws']
sizes = [home_wins, away_wins, draws]
colors = ['lightblue', 'lightcoral', 'lightgray']
plt.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
plt.title('Match Outcome Distribution')
plt.show()

print(f"Home wins: {home_wins} ({home_wins/len(df)*100:.1f}%)")
print(f"Away wins: {away_wins} ({away_wins/len(df)*100:.1f}%)")
print(f"Draws: {draws} ({draws/len(df)*100:.1f}%)")

In [ ]:
# Score distribution
plt.figure(figsize=(10, 6))
df['total_goals'].hist(bins=range(0, 15), edgecolor='black', alpha=0.7)
plt.title('Distribution of Total Goals per Match')
plt.xlabel('Total Goals')
plt.ylabel('Frequency')
plt.show()

## 7. Gap Analysis & Data Quality Issues

In [ ]:
# Identify potential data quality issues
print("DATA QUALITY ISSUES IDENTIFIED:")
print("="*50)

# Check for invalid scores
invalid_scores = df[(df['home_score'] < 0) | (df['away_score'] < 0)]
print(f"1. Matches with negative scores: {len(invalid_scores)}")

# Check for extremely high scores
high_scores = df[(df['home_score'] + df['away_score']) > 20]
print(f"2. Matches with more than 20 total goals: {len(high_scores)}")

# Check for teams that only appear once
all_teams = pd.concat([df['home_team'], df['away_team']])
single_appearance = all_teams.value_counts()[all_teams.value_counts() == 1]
print(f"3. Teams appearing only once: {len(single_appearance)}")

# Check for inconsistent country/city relationships
city_country = df.groupby('city')['country'].nunique()
inconsistent_cities = city_country[city_country > 1]
print(f"4. Cities with multiple country assignments: {len(inconsistent_cities)}")

# Check for future dates
future_matches = df[df['date'] > datetime.now()]
print(f"5. Matches with future dates: {len(future_matches)}")

In [ ]:
# Temporal gaps analysis
df_sorted = df.sort_values('date')
date_diffs = df_sorted['date'].diff()
large_gaps = date_diffs[date_diffs > pd.Timedelta(days=365)]

print("TEMPORAL GAPS (greater than 1 year):")
print("="*50)
for idx, gap in large_gaps.items():
    prev_date = df_sorted.loc[idx-1, 'date'] if idx > 0 else 'N/A'
    curr_date = df_sorted.loc[idx, 'date']
    print(f"Gap of {gap.days} days between {prev_date} and {curr_date}")

## 8. Summary & Recommendations

In [ ]:
print("EDA SUMMARY & RECOMMENDATIONS")
print("="*50)
print(f"Dataset contains {df.shape[0]} matches spanning from {df['date'].min().year} to {df['date'].max().year}")
print(f"Total unique teams: {len(pd.concat([df['home_team'], df['away_team']]).unique())}")
print(f"Total unique countries: {df['country'].nunique()}")
print(f"Total unique tournaments: {df['tournament'].nunique()}")
print(f"\nHome advantage: {home_wins/len(df)*100:.1f}% home win rate")
print(f"Average goals per match: {df['total_goals'].mean():.2f}")

print("\nRECOMMENDATIONS FOR results.py:")
print("1. Add data validation checks for scores")
print("2. Implement handling for potential missing values")
print("3. Add temporal gap detection")
print("4. Include team/country consistency checks")
print("5. Add statistical significance tests for home advantage")
print("6. Implement outlier detection for unusual scores")